In [147]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
import requests
import pandas as pd

In [148]:
uri = os.getenv('SBS_V1_MONGO_URI')

client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']

try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

nba_games_historical_collection = db['nba_games_historical']
nba_team_aggregated_game_stats_historical_collection = db['nba_team_aggregated_game_stats_historical']
nba_game_player_stats_historical_collection = db['nba_game_player_stats_historical']
nba_player_aggregated_game_stats_historical_collection = db['nba_player_aggregated_game_stats_historical']
cached_web_api_response_collection = db['cached_web_api_response']

Pinged your deployment. You successfully connected to MongoDB!


In [149]:
#########################################################
# get_event_odds ########################################
def get_event_odds(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/events/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']
    return response
#########################################################

#########################################################
# get_event_odds ########################################
#[derive(Debug, Deserialize, Clone)]
#[serde(rename_all = "camelCase")]
# pub struct GetOddsRequest {
#     pub sports: OddsApiSports,
#     pub regions: OddsApiRegions,
#     pub markets: Vec<String>,
#     pub odds_format: OddsFormat,
#     pub bookmakers: Vec<Bookmakers>
# }
def get_odds(req):
    url = 'https://sportsbettingsandboxapi.com/odds-api/odds/get'
    response = requests.post(url, json=req).json()['data']['events']
    return response
#########################################################

In [170]:
nba_events = get_event_odds('BasketballNba')

get_nba_odds_req = {
    'sports': 'BasketballNba',
    'regions': 'US',
    'markets': ['totals'],
    'oddsFormat': 'American',        
    'bookmakers': ['DraftKings']
}

nba_odds = get_odds(get_nba_odds_req)
nba_odds = list(filter(lambda x: len(x['bookmakers']) > 0, nba_odds))

for odds in nba_odds:
    odds['bookmakers'] = odds['bookmakers'][0]

event_market_map = dict()

for odds in nba_odds:
    market_map = dict()
    for market in odds['bookmakers']['markets']:
        market_map[market['key']] = market['outcomes'] 
    event_market_map[odds['id']] = market_map

In [171]:
def past_occurrence_strategy(event_market_map):
    

_IncompleteInputError: incomplete input (482210615.py, line 2)